# Validate expression split

This notebook is for manual validation of the rule-based relational versus attribute split produced by `data/classify_expressions.py`. Following the project methodology, review an approximately balanced audit sample of about 100 expressions per predicted class, record whether each prediction is correct, and use the resulting judgments to estimate precision for the `relational` and `attribute` labels.

In [ ]:
import os
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "configs" / "config.yaml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if not (repo_root / "configs" / "config.yaml").exists():
    raise RuntimeError("Could not locate repo root (configs/config.yaml not found in any parent directory).")
os.chdir(repo_root)
print("Working directory set to:", repo_root)

In [ ]:
from pathlib import Path

import pandas as pd

from common.utils import load_config

# Edit these before running the audit.
LOG_CSV_PATH = Path('data/splits/refcoco_val_classification_log.csv')
N_PER_LABEL = 100
RANDOM_SEED = 7

df = pd.read_csv(LOG_CSV_PATH)
sampled = pd.concat(
    [
        df[df['label'] == lbl].sample(n=min(N_PER_LABEL, (df['label'] == lbl).sum()), random_state=RANDOM_SEED)
        for lbl in sorted(df['label'].unique())
    ],
    ignore_index=True,
)

log_stem = LOG_CSV_PATH.name.removesuffix('_classification_log.csv')
dataset, split = log_stem.rsplit('_', 1)
config = load_config('configs/config.yaml')
ground_truth = (
    pd.read_json(Path('data/processed') / f'{dataset}_{split}.jsonl', lines=True)
    [['ref_id', 'file_name', 'bbox_xyxy']]
    .drop_duplicates(subset='ref_id')
    .set_index('ref_id')
)
sampled = sampled.merge(ground_truth, how='left', left_on='ref_id', right_index=True)

print(sampled['label'].value_counts().sort_index())
sampled.head()

In [ ]:
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

judgments = []

# Review one prediction at a time and record whether the predicted label is correct.
for row in sampled.itertuples(index=False):
    print()

    if pd.notna(row.file_name):
        image_path = Path(config['paths']['coco_images_dir']) / row.file_name
        try:
            with Image.open(image_path) as image:
                annotated_image = image.copy()
            ImageDraw.Draw(annotated_image).rectangle(row.bbox_xyxy, outline='red', width=3)
            plt.imshow(annotated_image)
            plt.axis('off')
            plt.show()
        except (FileNotFoundError, OSError):
            print(f"no image available for ref_id={row.ref_id}")
    else:
        print(f"no image available for ref_id={row.ref_id}")

    print(f"Expression: {row.expression}")
    print(f"Matched cue: {row.matched_cue}")
    print(f"Predicted label: {row.label}")

    while True:
        answer = input('Correct? [y/n]: ').strip().lower()
        if answer in {'y', 'yes', 'n', 'no'}:
            judgments.append(answer.startswith('y'))
            break
        print('Please answer with y or n.')

sampled = sampled.copy()
sampled['human_correct'] = judgments
sampled[['expression', 'matched_cue', 'label', 'human_correct']].head()

In [ ]:
from pathlib import Path

# Precision is the share of predicted examples judged correct within each label.
precision = sampled.groupby('label')['human_correct'].mean().sort_index()
for label, value in precision.items():
    print(f"{label}: {value:.3f}")

output_path = Path('results/expression_classifier_audit.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
sampled.to_csv(output_path, index=False)
print(f"Saved audit annotations to {output_path}")

Paste the resulting per-class precision numbers into the README `Limitations` section after finishing the audit.